#### Olist E-Commerce: Data CleaningThis notebook cleans the raw Olist Brazilian e-commerce dataset (9 CSV files, Sept 2016 - Oct 2018)ahead of loading it into Power BI for dashboarding. It handles missing values, drops unused columns,fixes data types, and exports the cleaned tables as CSVs.**Source:** Olist Store dataset (public, Kaggle) - a Brazilian e-commerce marketplace enabler.**Scope:** This notebook covers cleaning only. Business-question analysis was done separately in SQL(see `olist_sql_analysis.sql` in this repo).

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#### Load the raw dataAll 9 raw tables are loaded from Google Drive (Colab environment). If reproducing this notebookoutside Colab, replace the `drive.mount(...)` step and adjust file paths to wherever the rawOlist CSVs are stored locally.

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')
customers=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_customers_dataset.csv')
geolocation=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_geolocation_dataset.csv')
order_items=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_order_items_dataset.csv')
payments=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_order_payments_dataset.csv')
order_reviews=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_order_reviews_dataset.csv')
orders=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_orders_dataset.csv')
products=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_products_dataset.csv')
sellers=pd.read_csv('/content/drive/MyDrive/Dataset/olist/olist_sellers_dataset.csv')
product_category=pd.read_csv('/content/drive/MyDrive/Dataset/olist/product_category_name_translation.csv')

Mounted at /content/drive


#### Initial explorationEach raw table was inspected individually with `.head()`, `.shape`, `.info()`, `.describe()`,and `.duplicated().sum()` to understand structure, data types, and check for exact duplicate rows.`order_items`, `payments`, `products`, `sellers`, `order_reviews`, and `product_category` had noduplicate rows and no structural issues requiring action. Two checks did surface real findings,detailed below.

#### Check: does customer_unique_id map cleanly to customer_id?Olist assigns a new `customer_id` to each order, even for the same person - so `customer_id` isNOT a stable per-customer identifier. Checking for duplicate `customer_unique_id` values confirmsthis: the same real customer appears multiple times under different `customer_id` values. Thismeans any per-customer aggregation (revenue, order count, etc.) must group by `customer_unique_id`,never `customer_id`.

In [ ]:
customers[customers.duplicated('customer_unique_id',keep=False)].sort_values('customer_unique_id')

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
22066,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
...,...,...,...,...,...
75057,1ae563fdfa500d150be6578066d83998,ff922bdd6bafcdf99cb90d7f39cea5b3,17340,barra bonita,SP
27992,bec0bf00ac5bee64ce8ef5283051a70c,ff922bdd6bafcdf99cb90d7f39cea5b3,17340,barra bonita,SP
79859,d064be88116eb8b958727aec4cf56a59,ff922bdd6bafcdf99cb90d7f39cea5b3,17340,barra bonita,SP
64323,4b231c90751c27521f7ee27ed2dc3b8f,ffe254cc039740e17dd15a5305035928,37640,extrema,MG


#### Check for missing values`orders` has nulls in three date columns. `order_approved_at` and `order_delivered_carrier_date`nulls mostly correspond to orders that were never approved/shipped (cancelled, unavailable, etc.),which is expected. `order_delivered_customer_date` nulls need closer attention - some of theseare marked `order_status = 'delivered'` despite having no delivery date, which is a genuine datainconsistency, not an expected null.

In [ ]:
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


#### Investigate the "delivered but no delivery date" ordersBefore dropping anything, the specific anomalous orders were traced across related tables toconfirm they're genuinely broken records rather than a quirk of the join. Narrowing to ordersthat are `delivered` with a missing `order_delivered_customer_date` finds a small number of cases;narrowing further to orders where the carrier date is ALSO missing isolates the clearest anomaly -an order marked delivered that was apparently never even handed to the carrier. Checking thatorder's items, payments, and reviews confirms it has otherwise normal-looking data (a real order,real payment, real items) - it's specifically the delivery-tracking fields that are broken, notthe whole record. This supports dropping it rather than trying to impute a delivery date.

In [ ]:
orders[
    orders["order_delivered_customer_date"].isnull()
]["order_status"].value_counts()

,count
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


In [ ]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [ ]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isnull()) &
    (orders["order_delivered_customer_date"].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


In [ ]:
orders.loc[92643, "order_id"]

'2d858f451373b04fb5c984a1cc2defaf'

In [ ]:
order_items[
    order_items["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
19838,2d858f451373b04fb5c984a1cc2defaf,1,30b5b5635a79548a48d04162d971848f,f9bbdd976532d50b7816d285a22bd01e,2017-06-04 23:30:16,179.0,15.0


In [ ]:
payments[
payments["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
47287,2d858f451373b04fb5c984a1cc2defaf,1,credit_card,4,194.0


In [ ]:
order_reviews[
    order_reviews["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
51996,4e755f114e50d33b9ac6a56e0d7d3ea9,2d858f451373b04fb5c984a1cc2defaf,5,NaN,NaN,2017-06-25 00:00:00,2017-06-27 01:49:04


#### Fix: drop anomalous "delivered but no delivery date" rowsSince "delivered" should always have a delivery date, all rows matching this pattern (not justthe one traced above) are treated as a data error rather than a legitimate delivered order, andare dropped. This affects a negligible share of the ~99K orders.

In [ ]:
orders = orders[
    ~((orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isnull()))
].reset_index(drop=True)

#### Clean the products table- Missing `product_category_name` is filled with `'unknown'` rather than dropped, since the order  itself is still valid - we just don't know its category.- Physical dimension columns (weight, dimensions, photo count, name/description length) are dropped  since they aren't used in this analysis.

In [ ]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

products = products.drop(columns=[
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
])

#### Clean the reviews tableFree-text review comment columns are dropped - this analysis focuses on `review_score` and deliverytiming, not text/sentiment analysis of comments.

In [ ]:
order_reviews = order_reviews.drop(columns=["review_comment_title", "review_comment_message"])

#### Convert date columns to proper datetime typeAll order-related and review-related timestamp columns are converted from raw strings to`datetime64`, so delivery-time calculations and time-based grouping work correctly downstream.

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders[order_date_cols] = orders[order_date_cols].apply(pd.to_datetime)

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"])
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"])

orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


#### Export cleaned tablesEach cleaned table is exported to CSV for import into Power BI. Note: this export happens**before** any business-logic columns (like delivery on-time/late flags) are added - those arecalculated directly in Power BI via DAX measures instead, filtered explicitly to`order_status == 'delivered'` to avoid misclassifying orders that were never delivered.

In [ ]:
orders.to_csv('orders_clean.csv', index=False)
order_items.to_csv('order_items_clean.csv', index=False)
payments.to_csv('payments_clean.csv', index=False)
order_reviews.to_csv('order_reviews_clean.csv', index=False)
products.to_csv('products_clean.csv', index=False)
customers.to_csv('customers_clean.csv', index=False)
sellers.to_csv('sellers_clean.csv', index=False)
product_category.to_csv('product_category_clean.csv', index=False)
geolocation.to_csv('geolocation_clean.csv', index=False)